[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Kuo/Data-Engineering-Fork-of-AmFam-Workshop/blob/main/explorations/synthetic_policy_rag_walkthrough.ipynb)

# Synthetic mini-policy RAG walkthrough

**Fictional policy text** in `synthetic_data/mini_auto_policy.md`. No API keys required for retrieval + simple answer synthesis from top chunks.

**What you learn:** How policy Q&A maps to chunking, embeddings, and retrieval — the same shape as real coverage bots, without real customer data.

In [ ]:
import re
from pathlib import Path
import chromadb

ROOT = Path('..')
policy_path = ROOT / 'synthetic_data' / 'mini_auto_policy.md'
text = policy_path.read_text(encoding='utf-8')
# Chunk by ## headers
parts = re.split(r'\n##\s+', text)
chunks = [parts[0].strip()] + [f"## {p.strip()}" for p in parts[1:] if p.strip()]
print(len(chunks), 'chunks')
for i, c in enumerate(chunks[:3]):
    print(f'--- chunk {i} ---\n', c[:200], '...')

## Build a local vector store (ChromaDB default embedding)

In [ ]:
import os
out = ROOT / 'outputs' / 'chroma_synthetic_policy'
out.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(out))
try:
    client.delete_collection('synthetic_policy')
except Exception:
    pass
col = client.create_collection('synthetic_policy', metadata={'hnsw:space': 'cosine'})
ids = [f'chunk_{i}' for i in range(len(chunks))]
col.add(documents=chunks, ids=ids)
print('Indexed', len(chunks), 'chunks')

## Ask coverage-style questions

In [ ]:
questions = [
    'What is my collision deductible?',
    'How much per day for a rental car?',
    'Is glass repair covered without deductible?',
    'How far will towing cover?',
    'Does collision cover commercial racing?',
]
for q in questions:
    r = col.query(query_texts=[q], n_results=2, include=['documents', 'distances'])
    print('\nQ:', q)
    print('Top chunk (excerpt):', r['documents'][0][0][:400].replace('\n', ' '), '...')
    print('Distance (lower=better):', round(r['distances'][0][0], 4))

## What we learned

- **Chunking matters** — Article boundaries (`##`) align with user mental model of "sections."
- **Retrieval distance** — Low distance = high confidence the right clause was found; still not proof the *answer* is correct (need judge / human for production).
- **Insurance mapping** — Same pipeline scales to real policy PDFs + endorsements; your `rag_hallucination_scoring` notebook adds the **governance layer** (citation + judge + tier).
- **NAIC narrative** — Documented retrieval + logging (see `rag_eval_logging.ipynb`) supports audit trails.